# Содержание
* Цель и продуктовые вопросы
* Ключевые решения и определения метрик
* Данные и ограничения
* Импорт библиотек
* Чтение данных
* Быстрая проверка структуры и качества
* Подготовка полей для воронки
* Базовая воронка
* Воронка в разрезах
* Воронка по категориям
* Структура покупок по категориям
* Анализ рейтинга по категориям
* Выводы и рекомендации

# Цель и продуктовые вопросы

__Цель анализа__ – понять, где и у каких сегментов пользователей воронки e-commerce магазина: page_view → add_to_cart → checkout → purchase работает заметно хуже, а также проверить, есть ли категории с систематически низкими оценками.

__Продуктовые вопросы:__

__Вопрос 1 (основной):__
Где и у кого воронка page_view → add_to_cart → checkout → purchase работает заметно лучше или хуже?
Конкретно нас интересует:
* По странам
* По возрастным группам
* По устройствам
* По категориям товаров

__Зачем этот вопрос:__
Найти сегменты с сильной просадкой на каком-то этапе воронки. Именно там обычно лежат самые большие возможности для роста (или самые серьёзные проблемы продукта).


__Вопрос 2 (вторичный, но важный):__
Есть ли категории товаров, которые систематически получают заметно более низкие оценки?

__Зачем:__
Понять, связано ли это: с качеством самого товаром; с ожиданиями пользователей или; возможно, с тем, как система рекомендаций/поиска подбирает товары.


# Ключевые решения и определения метрик

__Единица наблюдения__ \
Воронку считаем __по сессиям__.

Сессия считается достигшей определённого этапа воронки,
если в ней есть __хотя бы одно__ событие соответствующего типа:
* page_view
* add_to_cart
* checkout
* purchase

Почему сессии, а не пользователи:
нас в первую очередь интересует поведение внутри визита и места,
где пользователи отваливаются в конкретном сценарии использования продукта.

__Определение метрик воронки__
* __Conversion rate между шагами__ =
  количество сессий, дошедших до шага N+1 / количество сессий, дошедших до шага N

* __Общий conversion rate__ =
  количество сессий с purchase / количество сессий с page_view

__Возрастные группы__
Возраст разобьём на следующие группы:
* 18–24
* 25–34
* 35–44
* 45–54
* 55+

Слишком мелкую нарезку делать не будем, чтобы не получить шумные и маленькие сегменты.

__Источник device__ \
Для воронки по сессиям используем `sessions.device`
(поле заполнено полностью, расхождений с orders по наличию данных нет).

__Связь события и категории товара__
Категорию товара всегда берём через связку:
`events.product_id` → `products.category`

То есть категория привязывается к конкретному событию,
а не считается «средней» по пользователю или сессии.


__Ограничение по product_id и работа с категориями__

* Полную воронку (все 4 шага) считаем без разреза по категориям.
* Разрез по категориям делаем только для перехода
  page_view → add_to_cart (где product_id есть).
* Структуру покупок по категориям смотрим через
  orders + order_items + products.
* Попытки восстановить категорию на этапе checkout
  не применяем, чтобы не вводить слабо обоснованные допущения.

__Обработка нескольких товаров в одной сессии__
Если в сессии есть несколько товаров,
сессия всё равно считается достигшей этапа
при наличии хотя бы одного события нужного типа.

(При необходимости позже можно дополнительно посмотреть
product-level воронку.)

# Данные и ограничения

__О датасете:__

Синтетические транзакции электронной коммерции + кликстрим (2020-2025)
Полностью синтетический мультитабельный набор данных, моделирующий интернет-магазин: клиенты продукты, сессии, события кликстевого потока, заказы, элементы заказа и отзывы.
Построено с помощью фейкеров и эвристических воронок, чтобы они напоминали реальный алгоритм просмотра и покупки.

Источник данных kaggle: https://www.kaggle.com/datasets/wafaaelhusseini/e-commerce-transactions-clickstream?resource=download

__Таблицы:__
* customers.csv — customer profiles, signup dates, country, opt-in
* products.csv — catalog with categories, prices, costs, margins
* sessions.csv — session metadata (device, source, start time, country)
* events.csv — page_view / add_to_cart / checkout / purchase with timestamps
* orders.csv — order headers (payment, discount, totals)
* order_items.csv — line items (quantity, unit price, line total)
* reviews.csv — product ratings & short text reviews

__Ключевые ограничения, которые уже видны:__
* Отзывы оставляют не все покупатели.
* Возможны расхождения device между sessions и orders.
* Данные синтетические (если это важно для интерпретации).
* __В таблице events поле `product_id` полностью отсутствует
  на этапах `checkout` и `purchase`.__
  Из-за этого невозможно напрямую считать воронку
  в разрезе категорий на последних двух шагах.

__ER–диаграмма__
![docs/ER–diagram](docs/ER–diagram.png)

# Импорт библиотек

In [2]:
import pandas as pd

# Чтение данных

In [3]:
# Чтение всех таблиц
customers = pd.read_csv('tables/customers.csv')
events = pd.read_csv('tables/events.csv')

order_items = pd.read_csv('tables/order_items.csv')
orders = pd.read_csv('tables/orders.csv')

products = pd.read_csv('tables/products.csv')
reviews = pd.read_csv('tables/reviews.csv')
sessions = pd.read_csv('tables/sessions.csv')

# Быстрая проверка структуры и качества

Быстрая проверка (критично для воронки)

Проверяем только то, что влияет на достоверность conversion rate
и возможность считать воронку в нужных разрезах.

In [4]:
# Подтверждаем, что в данных есть только ожидаемые этапы воронки
events['event_type'].unique()

<StringArray>
['page_view', 'add_to_cart', 'checkout', 'purchase']
Length: 4, dtype: str

In [5]:
# Проверка пропусков в данных
# Доля пропусков product_id по этапам воронки
events.groupby('event_type')['product_id'].apply(lambda x: x.isnull().mean())

event_type
add_to_cart    0.0
checkout       1.0
page_view      0.0
purchase       1.0
Name: product_id, dtype: float64

In [6]:
# Проверка пропусков в данных
print('sessions.device nulls:', sessions['device'].isnull().sum())
print('orders.device nulls:', orders['device'].isnull().sum())

sessions.device nulls: 0
orders.device nulls: 0


In [7]:
# Проверка пропусков в данных
customers['age'].isnull().sum()

np.int64(0)

In [8]:
customers['age'].describe()

count    20000.000000
mean        46.492550
std         16.767961
min         18.000000
25%         32.000000
50%         46.500000
75%         61.000000
max         75.000000
Name: age, dtype: float64

In [9]:
# Проверка пропусков в данных
sessions['country'].isnull().sum()

np.int64(0)

In [10]:
# Пятибалльный рейтинг
reviews['rating'].unique()

array([2, 4, 5, 3, 1])

In [11]:
print('reviews.rating nulls: ', reviews['rating'].isnull().sum())
print('reviews.review_text nulls: ', reviews['review_text'].isnull().sum())

reviews.rating nulls:  0
reviews.review_text nulls:  0


* Этапы воронки на месте, объёмы адекватные.
* product_id отсутствует на checkout и purchase → категорийный разрез на этих шагах через events невозможен.
* device в sessions заполнен полностью → берём его.
* Возраст чистый → группы можно создавать.
* Страны заполнены полностью.
* Колонки рейтинга и отзыва заполнены полностью.

# Подготовка полей для воронки (возрастные группы, выбор device, джойны категорий и т.д.)

In [46]:
# Используем 17, так как pd.cut по умолчанию не включает левую границу (нужно захватить 18 лет)
customers['age_group'] = pd.cut(
    customers['age'],
    bins=[17, 24, 34, 44, 54, 100],
    labels=['18-24', '25-34', '35-44', '45-54', '55+']
)

# Собираем сырые данные для воронки в единую таблицу
joined_table = (
    customers
    .merge(sessions, how='inner', on='customer_id', suffixes=('_customer', '_session'))
    .merge(events, how='inner', on='session_id')
)

# Разворачиваем типы событий в логические колонки (флаги)
event_dummies = pd.get_dummies(joined_table['event_type']).astype(bool)
joined_table = pd.concat([joined_table, event_dummies], axis=1)

# Агрегируем до уровня сессии, чтобы определить факт прохождения каждого шага воронки
session_funnel = (
    joined_table
    .groupby(['session_id', 'age_group', 'device', 'country_session'], as_index=False)
    .agg(
        has_page_view=('page_view', 'max'),
        has_add_to_cart=('add_to_cart', 'max'),
        has_checkout=('checkout', 'max'),
        has_purchase=('purchase', 'max')
    )
).reset_index()
session_funnel

,index,session_id,age_group,device,country_session,has_page_view,has_add_to_cart,has_checkout,has_purchase
0,0,1,35-44,mobile,DE,True,True,False,False
1,1,2,55+,desktop,PL,True,True,True,True
2,2,3,45-54,tablet,FR,True,True,True,True
3,3,4,18-24,mobile,GB,True,False,False,False
4,4,5,18-24,desktop,ES,True,False,False,False
...,...,...,...,...,...,...,...,...,...
119995,119995,119996,35-44,mobile,NL,True,True,True,False
119996,119996,119997,55+,mobile,AU,True,True,True,False
119997,119997,119998,55+,tablet,PL,True,False,False,False
119998,119998,119999,55+,mobile,US,True,True,True,True


# Базовая воронка (без срезов)

In [47]:
# Абсолютные значения по шагам (в порядке воронки)
STEPS = ['page_view', 'add_to_cart', 'checkout', 'purchase']
ABSOLUTE_VALUES = [
    session_funnel['has_page_view'].sum(),
    session_funnel['has_add_to_cart'].sum(),
    session_funnel['has_checkout'].sum(),
    session_funnel['has_purchase'].sum()
]

# Собираем таблицу
funnel_base = pd.DataFrame({
    'steps': STEPS, 'sessions': ABSOLUTE_VALUES
})

# Конверсия от предыдущего шага
funnel_base['cr_from_previous'] = (
        funnel_base['sessions'] / funnel_base['sessions'].shift(1)
)

# Для первого шага ставим 1.0 (или 100%)
funnel_base.loc[0, 'cr_from_previous'] = 1

# Конверсия от начала воронки
funnel_base['cr_from_start'] = (
    funnel_base['sessions'] / funnel_base['sessions'].iloc[0]
)

funnel_base

,steps,sessions,cr_from_previous,cr_from_start
0,page_view,120000,1.000000,1.000000
1,add_to_cart,81518,0.679317,0.679317
2,checkout,44909,0.550909,0.374242
3,purchase,33580,0.747734,0.279833


# Воронка в разрезах

In [14]:
# Воронка по странам
funnel_country = (
    session_funnel
    .groupby('country_session')
    .agg({
        'has_page_view': 'sum',
        'has_add_to_cart': 'sum',
        'has_checkout': 'sum',
        'has_purchase': 'sum',
    })
).reset_index()

# Конверсия page_view –> add_to_cart
funnel_country['cr_view_to_cart'] = (
    funnel_country['has_add_to_cart'] / funnel_country['has_page_view']
)

# Конверсия add_to_cart –> checkout
funnel_country['cr_cart_to_checkout'] = (
    funnel_country['has_checkout'] / funnel_country['has_add_to_cart']
)

# Конверсия checkout –> purchase
funnel_country['cr_checkout_to_purchase'] = (
    funnel_country['has_purchase'] / funnel_country['has_checkout']
)

# Конверсия page_view –> purchase
funnel_country['cr_overall'] = (
    funnel_country['has_purchase'] / funnel_country['has_page_view']
)

funnel_country

,country_session,has_page_view,has_add_to_cart,has_checkout,has_purchase,cr_view_to_cart,cr_cart_to_checkout,cr_checkout_to_purchase,cr_overall
0,AE,3424,2368,1244,913,0.691589,0.525338,0.733923,0.266647
1,AU,6443,4367,2413,1797,0.677790,0.552553,0.744716,0.278907
2,BR,8573,5841,3287,2464,0.681325,0.562746,0.749620,0.287414
3,CA,6057,4064,2209,1651,0.670959,0.543553,0.747397,0.272577
4,DE,8332,5681,3116,2321,0.681829,0.548495,0.744865,0.278565
5,ES,5580,3821,2086,1558,0.684767,0.545930,0.746884,0.279211
6,FR,7811,5309,2921,2188,0.679682,0.550198,0.749059,0.280118
7,GB,9451,6406,3539,2653,0.677812,0.552451,0.749647,0.280711
8,IN,9567,6471,3509,2642,0.676388,0.542265,0.752921,0.276158
9,JP,5841,3976,2198,1655,0.680705,0.552817,0.752957,0.283342


In [15]:
# Воронка по возрастным группам
funnel_age_group = (
    session_funnel
    .groupby('age_group')
    .agg({
        'has_page_view': 'sum',
        'has_add_to_cart': 'sum',
        'has_checkout': 'sum',
        'has_purchase': 'sum',
    })
).reset_index()

# Конверсия page_view –> add_to_cart
funnel_age_group['cr_view_to_cart'] = (
        funnel_age_group['has_add_to_cart'] / funnel_age_group['has_page_view']
)

# Конверсия add_to_cart –> checkout
funnel_age_group['cr_cart_to_checkout'] = (
        funnel_age_group['has_checkout'] / funnel_age_group['has_add_to_cart']
)

# Конверсия checkout –> purchase
funnel_age_group['cr_checkout_to_purchase'] = (
        funnel_age_group['has_purchase'] / funnel_age_group['has_checkout']
)

# Конверсия page_view –> purchase
funnel_age_group['cr_overall'] = (
    funnel_age_group['has_purchase'] / funnel_age_group['has_page_view']
)

funnel_age_group

,age_group,has_page_view,has_add_to_cart,has_checkout,has_purchase,cr_view_to_cart,cr_cart_to_checkout,cr_checkout_to_purchase,cr_overall
0,18-24,14296,9759,5339,4012,0.682639,0.547085,0.751452,0.280638
1,25-34,20946,14164,7865,5865,0.676215,0.555281,0.745709,0.280006
2,35-44,20395,13895,7621,5696,0.681294,0.548471,0.747408,0.279284
3,45-54,20777,14053,7752,5797,0.676373,0.551626,0.747807,0.279010
4,55+,43586,29647,16332,12210,0.680195,0.550882,0.747612,0.280136


In [16]:
# Воронка по устройствам
funnel_device = (
    session_funnel
    .groupby('device')
    .agg({
        'has_page_view': 'sum',
        'has_add_to_cart': 'sum',
        'has_checkout': 'sum',
        'has_purchase': 'sum',
    })
).reset_index()

# Конверсия page_view –> add_to_cart
funnel_device['cr_view_to_cart'] = (
        funnel_device['has_add_to_cart'] / funnel_device['has_page_view']
)

# Конверсия add_to_cart –> checkout
funnel_device['cr_cart_to_checkout'] = (
        funnel_device['has_checkout'] / funnel_device['has_add_to_cart']
)

# Конверсия checkout –> purchase
funnel_device['cr_checkout_to_purchase'] = (
        funnel_device['has_purchase'] / funnel_device['has_checkout']
)

# Конверсия page_view –> purchase
funnel_device['cr_overall'] = (
        funnel_device['has_purchase'] / funnel_device['has_page_view']
)

funnel_device

,device,has_page_view,has_add_to_cart,has_checkout,has_purchase,cr_view_to_cart,cr_cart_to_checkout,cr_checkout_to_purchase,cr_overall
0,desktop,45547,30990,17069,12750,0.680396,0.550791,0.746968,0.279931
1,mobile,65942,44800,24695,18469,0.679385,0.551228,0.747884,0.280079
2,tablet,8511,5728,3145,2361,0.673011,0.549057,0.750715,0.277406


__Промежуточные выводы по сегментам__

* Значимых различий в conversion rate по странам, возрасту и устройствам нет.
* Главный провал воронки — шаг add_to_cart → checkout (~45% отвала).
* Общий conversion page_view → purchase ≈ 28%.

# Воронка по категориям

In [17]:
# Воронка по категориям
# Берём только события, где есть product_id и которые нужны для ранней воронки
events_cat = events[events['event_type'].isin(['page_view', 'add_to_cart'])].copy()

# Добавляем категорию
events_cat = events_cat.merge(
    products[['product_id', 'category']],
    on='product_id',
    how='inner'  # inner — чтобы не тащить события без категории
)

# Разворачиваем типы событий в логические колонки (флаги)
events_cat['is_page_view'] = events_cat['event_type'] == 'page_view'
events_cat['is_add_to_cart'] = events_cat['event_type'] == 'add_to_cart'

session_category = (
    events_cat
    .groupby(['session_id', 'category'], as_index=False)
    .agg(
        has_page_view=('is_page_view', 'max'),   # был ли хотя бы один view
        has_add_to_cart=('is_add_to_cart', 'max') # был ли хотя бы один cart
    )
)

funnel_category = (
    session_category
    .groupby('category', as_index=False)
    .agg(
        has_page_view=('has_page_view', 'sum'),
        has_add_to_cart=('has_add_to_cart', 'sum')
    )
)

# Конверсия page_view –> add_to_cart
funnel_category['cr_view_to_cart'] = (
    funnel_category['has_add_to_cart'] / funnel_category['has_page_view']
)

funnel_category.sort_values(by='cr_view_to_cart', ascending=False)

,category,has_page_view,has_add_to_cart,cr_view_to_cart
1,Books,52234,24197,0.463242
6,Toys,49441,21673,0.438361
0,Beauty,46724,19376,0.414691
3,Fashion,41373,15214,0.367728
4,Home & Kitchen,40258,14474,0.359531
5,Sports,38037,12665,0.332965
2,Electronics,30362,7283,0.239872


__Промежуточные итоги по категориям__
* По категориям (page_view → add_to_cart)
* Есть существенные различия между категориями.
* Лучше всего конвертят: Books (\~46%), Toys (\~44%), Beauty (\~42%).
* Хуже всего: Electronics (~24%) — почти в 2 раза ниже лидеров.
* Sports, Fashion, Home & Kitchen — в середине (33–37%).
* В отличие от страны/возраста/устройства, категория товара заметно влияет на вероятность добавления в корзину.

# Структура покупок по категориям

In [18]:
# inner join: оставляем только те позиции, у которых есть
# и продукт, и заказ (битые связи отбрасываем)
joined_table = (
    products
    .merge(order_items, on='product_id', how='inner')
    .merge(orders, on='order_id', how='inner')
)

# Агрегируем до уровня категории
category_distribution = (
    joined_table
    .groupby('category')
    .agg(
        category_revenue=('line_total_usd', 'sum'),
        items_count=('product_id', 'count'),
        orders_count=('order_id', 'nunique')
    )
).reset_index()

# Доля категории в общей выручке магазина
# Отвечает на вопрос: «какая часть денег приходит из этой категории?»
category_distribution['proportion_revenue'] = (
        category_distribution['category_revenue'] / joined_table['line_total_usd'].sum()
)

# Доля категории по числу купленных позиций
# Отвечает на вопрос: «как часто эту категорию покупают?»
# (независимо от цены)
category_distribution['proportion_item_count'] = (
    category_distribution['items_count'] / category_distribution['items_count'].sum()
)

# Доля категории по числу заказов, где она встречалась
# Важно: сумма этих долей может быть > 100%,
# потому что один заказ может содержать несколько категорий
category_distribution['proportion_orders_count'] = (
    category_distribution['orders_count'] / category_distribution['orders_count'].sum()
)

# Сортируем: сначала самые денежные категории
category_distribution = category_distribution.sort_values(
    by=['category_revenue', 'proportion_revenue'],
    ascending=[False, False]
)

category_distribution

,category,category_revenue,items_count,orders_count,proportion_revenue,proportion_item_count,proportion_orders_count
4,Home & Kitchen,840736.82,7299,5985,0.173864,0.123371,0.126205
5,Sports,832349.63,6346,5284,0.172129,0.107263,0.111423
3,Fashion,824739.33,7778,6267,0.170555,0.131467,0.132151
2,Electronics,692866.07,3455,3018,0.143284,0.058398,0.063640
0,Beauty,692312.49,10245,8055,0.143170,0.173166,0.169854
6,Toys,569153.08,11352,8878,0.117700,0.191877,0.187209
1,Books,383450.80,12688,9936,0.079297,0.214458,0.209519


__Промежуточные итоги по структуре покупок__

* По выручке лидируют Home & Kitchen, Sports, Fashion (~17% каждая).
* Books — чаще всего по числу позиций (\~21%), но дают меньше всего выручки (~8%): низкий средний чек при высокой частоте покупок.
* Electronics — умеренная доля выручки (\~14%) при самом низком числе позиций (~6%): выше средний чек, реже покупают.
* Toys и Beauty — заметны по количеству покупок, по деньгам — середина.

# Анализ рейтингка по категориям

In [45]:
joined_table = (
    products
    .merge(reviews, on='product_id', how='inner')
)

rating_distribution = (
    joined_table
    .groupby('category')
    .agg(
        n_reviews=('rating', 'count'),
        avg_rating=('rating', 'mean'),
        proportion_less_or_equal_2=('rating', lambda x: (x <= 2).mean()),
        proportion_greater_or_equal_4=('rating', lambda x: (x >= 4).mean())
    )
).reset_index()

rating_distribution

,category,n_reviews,avg_rating,proportion_less_or_equal_2,proportion_greater_or_equal_4
0,Beauty,1949,3.966136,0.104156,0.721396
1,Books,2341,3.916275,0.108501,0.701837
2,Electronics,691,3.866860,0.130246,0.672938
3,Fashion,1385,3.933574,0.115523,0.714801
4,Home & Kitchen,1344,3.946429,0.108631,0.709077
5,Sports,1113,3.960467,0.098832,0.707996
6,Toys,1957,3.925396,0.110884,0.702606


In [40]:
joined_table['rating'].value_counts(normalize=True)

rating
5    0.376160
4    0.330705
3    0.183673
2    0.070223
1    0.039239
Name: proportion, dtype: float64

__Промежуточные итоги по вопросу 2__
* Средние оценки по всем категориям близки (примерно 3.87–3.97).
* Доля низких оценок (1–2) тоже сопоставима (~10–13%).
* Electronics немного слабее остальных (ниже средний rating, выше доля 1–2), но разница невелика.
* Систематически «плохих» категорий по рейтингу не видно.
* Вероятная причина равномерности — синтетическая природа данных.
* Ограничение: отзывы оставляют не все покупатели; возможна смещённость (недовольные пишут чаще).

# Выводы и рекомендации

__Ответ на Вопрос 1: где проседает воронка__

**Общая воронка (по сессиям)**
- page_view → purchase ≈ 28%.
- Главный провал — add_to_cart → checkout: до оформления не доходит ~45% сессий, уже добавивших товар в корзину.
- Шаги view → cart (\~68%) и checkout → purchase (~75%) относительно стабильны.

**Сегменты: страна, возраст, устройство**
- Значимых различий в conversion rate нет (разброс в пределах долей процентного пункта).
- Точечные улучшения «под мобильных» или «под конкретную страну» на этих данных не приоритетны.
- Равномерность, скорее всего, связана с синтетической природой датасета.

**Категории товаров**
Полную 4-шаговую воронку по категориям построить нельзя (`product_id` отсутствует на checkout/purchase).
Смотрели два среза:

1. **page_view → add_to_cart (session + category)**
   - Лучше: Books (\~46%), Toys (\~44%), Beauty (~42%).
   - Хуже: Electronics (\~24%) — почти в 2 раза ниже лидеров.
   - Sports, Fashion, Home & Kitchen — середина (33–37%).
   - Категория — единственный срез, где различия заметны.

2. **Структура покупок**
   - По выручке: Home & Kitchen, Sports, Fashion.
   - Books — много позиций, мало выручки (низкий чек).
   - Electronics — меньше позиций, заметная выручка (выше чек).

**Связка поведения и денег**
- Electronics: слабо конвертит просмотр в корзину, но когда покупают — дороже; при этом оценок меньше, чем у других категорий.
- Books: легко кладут в корзину и часто покупают, но вклад в выручку небольшой.
- Home / Sports / Fashion: основной денежный объём при средней конверсии в корзину.

__Ответ на Вопрос 2: рейтинги по категориям__

- Средние rating по категориям близки (≈ 3.87–3.97).
- Доля оценок 1–2 ≈ 10–13%; доля 4–5 ≈ 67–72%.
- Electronics чуть слабее (ниже среднее, выше доля низких), но это не резкий провал.
- Систематически «токсичных» категорий по rating не видно.
- Объёмы отзывов достаточные у большинства категорий; у Electronics отзывов меньше (~690 против 1100–2300 у остальных) — выводы по нему чуть менее устойчивы.
- Ограничение: отзывы пишут не все; возможна смещённость (недовольные активнее оставляют feedback).

__Что это значит для продукта (в рамках данных)__

1. **Приоритет №1** — общий сценарий add_to_cart → checkout (не сегменты).
   Имеет смысл гипотезы про: цену доставки, неожиданные комиссии, UX оформления, обязательные поля, доверие к оплате.

2. **Приоритет №2** — Electronics на этапе «просмотр → корзина».
   Гипотезы: цена, сравнение характеристик, слабые карточки, потребность в большем research перед добавлением.

3. **Не размазывать усилия** по странам / возрасту / device — на этих данных сигнала нет.

4. **Рейтинги** не дают сильного рычага для приоритизации категорий; для понимания *почему* низкие оценки нужен текст отзывов.

__Ограничения анализа__

- Данные синтетические → слабая дифференциация по сегментам и рейтингам может быть артефактом генерации.
- Нет `product_id` на checkout/purchase → нельзя честно построить полную категорийную воронку.
- Отзывы покрывают не все покупки; тональность `review_text` не анализировалась.
- Воронка считалась по сессиям («хотя бы одно событие типа»); интенсивность кликов внутри сессии не разбиралась.

__Возможные следующие шаги__

1. Углубить этап cart → checkout: где именно отвал (если появятся более детальные события).
2. По Electronics — качественный разбор карточек, цен, сравнения с конкурентами.
3. Анализ тональности `review_text` (особенно для Electronics и категорий с чуть более высокой долей 1–2).